# Low-temperature DAC (LT-DAC)

LT-DAC ("low-temperature DAC") is a form of direct air capture that pulls CO2 out of the atmosphere using a solid sorbent and low-grade heat rather than electricity. The real-world process, temperature/vacuum-swing adsorption (TVSA) - the same technology class as Climeworks' commercial plants - cycles a bed of amine-functionalised sorbent between two phases: a fan draws ambient air through the bed, and CO2 binds to the sorbent at ambient temperature (adsorption); the bed is then sealed under vacuum and heated to a low regeneration temperature of roughly 80-120C - low enough to be driven by waste heat rather than dedicated fuel, unlike liquid-solvent DAC's 500-900C - which releases the CO2 and steam and regenerates the sorbent (desorption). The steam is condensed out, separating it from the CO2 gas product (Climeworks' published methodology reports >95% CO2 purity), and the cooled, regenerated sorbent is recycled back to the contactor for another cycle.

This repository does not cite a specific sorbent-chemistry paper for LT-DAC - its numbers are an internal waste-heat energy-balance assumption rather than one drawn from a named study (unlike E-DAC, which cites Sabatino et al. (2020) and Leonzio & Shah (2025); see `nbs/edac_diagram.ipynb`). Instead, LT-DAC is modelled as piggybacking on co-located gas-CCS generation:

1. **Heat-limited capture.** LT-DAC only runs when gas-CCS generation runs, and its capture is limited by *usable waste heat*, not electricity: fuel input is derived from gas output and the plant's electrical efficiency (0.55 for CCGT), a fraction of that fuel input is waste heat (0.45), a fraction of *that* is usable at low temperature (1.00 for CCGT), and the usable heat is divided by a fixed gas-input intensity of 1.1389 MWh per tonne CO2 to give gross capture (`src/assumptions.py` `LTDAC`, `GasTechDACParameters`; dispatch logic in `src/power_system_core_no_h2.py` lines 38-101, mirrored in `src/power_system.py` lines 239-253). An older, unused assumption of 4 GJ/t heat input is marked `Deprecated` in the code and superseded by this waste-heat derivation.
2. **Auxiliary electricity add-back.** Running LT-DAC's own fans and vacuum pumps draws a small amount of electricity (0.331 MWh/t); rather than treating this as a separate load, the model adds it back onto the gas-CCS plant's own generation, so it is accounted for as part of that plant's gross output.
3. **Net capture.** Because LT-DAC only runs alongside gas generation, and burning gas emits CO2, its *gross* capture is added to E-DAC's removals and then netted against the gas plant's own emissions (gas-CCS output times a 0.34 t/MWh emissions intensity) to get the net CO2 removed (`src/power_system.py` lines 316-318). Cost is a flat levelised rate: gross capture times an LCOR of 119 GBP/t (`src/assumptions.py` `LTDAC.LCOR`; a separate `LCOE` figure of 193 GBP/t is marked as an incorrect placeholder and not used).

The figure below shows the real-world TVSA process itself, laid out to fit a single manuscript column - matching the companion E-DAC figure.

![How low-temperature DAC (LT-DAC) works: ambient air is drawn through a solid amine sorbent bed, which binds CO2 at ambient temperature. The sorbent is then regenerated by sealing the bed under vacuum and heating it to a low temperature (~80-120C), releasing CO2 and steam. The steam is condensed out, separating it from the CO2 gas product, while the cooled, regenerated sorbent is recycled back to the contactor.](img/ltdac_diagram.svg)
